In [1]:
import subprocess
import pickle

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from scipy.stats import spearmanr, rankdata
from scipy.spatial.distance import pdist, squareform

from tqdm.notebook import tqdm

In [2]:
SEED = 0
N_PERM = 10_000

In [3]:
base = "/kaggle/input/notebooks/lukemiller1987/n14-aaai-dataset/"
data_dict_path = "n14_data_dict.pkl"
structure_key_path = "n14_structure_key_dict.pkl"
readme_path = "n14_readme.pkl"
with open(base+data_dict_path, 'rb') as f:
    data_dict = pickle.load(f)
with open(base+structure_key_path, 'rb') as f:
    structure_key_dict = pickle.load(f)
with open(base+readme_path, 'rb') as f:
    readme = pickle.load(f)
print(readme)

QuIC EXACT EMBEDDINGS — EXHAUSTIVE CONNECTED CUBIC GRAPHS, n=14

Generated: 2026-07-09
Producer:  quic-cycle-correlation-n14 (Kaggle)
Author:    Luke Miller (UMKC)
Context:   Decodability study of the QuIC quantum graph embedding
           (companion to QCE 2026 paper, arXiv:2604.18841).

CONTENTS
--------
n14_data_dict.pkl
    dict keyed by graph index (0..508), one entry per graph in the
    complete enumeration of connected 3-regular graphs on 14 vertices
    (509 graphs, generated by nauty-geng -c -d3 -D3 14, order as emitted).
    Per-entry fields:
      graph6         : str   — canonical graph6 string (reconstruct graph via
                               nx.from_graph6_bytes(s.encode()); circuit via
                               QuIK_circuit(adj_mat) with config below)
      adj_mat        : (14,14) float array — adjacency matrix
      num_triangles  : int   — simple 3-cycles (nx.simple_cycles, each once)
      num_4_cycles   : int   — simple 4-cycles
      num_5_cycles   : int

In [4]:
keys = sorted(data_dict)
vectors       = np.vstack([data_dict[k]['exact_vector'] for k in keys])
num_triangles = np.array([data_dict[k]['num_triangles'] for k in keys])
num_4_cycles  = np.array([data_dict[k]['num_4_cycles']  for k in keys])
num_5_cycles  = np.array([data_dict[k]['num_5_cycles']  for k in keys])

In [5]:
rng = np.random.default_rng(SEED)
rand_scalars = rng.random(len(keys))

emb_dist     = pdist(vectors, metric='cityblock')
tri_dist     = pdist(num_triangles.reshape(-1, 1), metric='cityblock')
cycle_4_dist = pdist(num_4_cycles.reshape(-1, 1),  metric='cityblock')
cycle_5_dist = pdist(num_5_cycles.reshape(-1, 1),  metric='cityblock')
rand_dist    = pdist(rand_scalars.reshape(-1, 1),  metric='cityblock')

In [6]:
def mantel_p(emb_dist, values, rho_obs, n_perm=N_PERM, seed=SEED):
    rng = np.random.default_rng(seed)
    hits = 0
    v = values.reshape(-1, 1)
    for _ in range(n_perm):
        d = pdist(v[rng.permutation(len(values))], metric='cityblock')
        r, _ = spearmanr(emb_dist, d)
        hits += (r >= rho_obs)
    return (hits + 1) / (n_perm + 1)

results = {}
for name, target_dist, target_vals in [
        ('triangles', tri_dist,     num_triangles),
        ('4-cycles',  cycle_4_dist, num_4_cycles),
        ('5-cycles',  cycle_5_dist, num_5_cycles),
        ('control',   rand_dist,    rand_scalars)]:
    rho, _ = spearmanr(emb_dist, target_dist)
    p = mantel_p(emb_dist, target_vals, rho)
    results[name] = {'rho': rho, 'p': p}
    print(f'{name:>10}:  rho = {rho:+.3f},  Mantel p = {p:.4f}')

 triangles:  rho = +0.903,  Mantel p = 0.0001
  4-cycles:  rho = +0.290,  Mantel p = 0.0001
  5-cycles:  rho = +0.109,  Mantel p = 0.0001
   control:  rho = +0.005,  Mantel p = 0.3152


In [7]:
# E1 level 1: within fixed-triangle strata, do C4 and C5 organize the geometry?
# Each stratum is an independent graph set -> Fisher's method combines per-stratum
# Mantel p-values into one gate decision number per target.

from scipy.stats import chi2
from numpy.linalg import svd

MIN_STRATUM = 15          # >= 105 pairs; below this the Mantel is underpowered
N_PERM_STRATUM = 10_000

def effective_rank(X):
    var = svd(X - X.mean(axis=0), full_matrices=False, compute_uv=False) ** 2
    return var.sum() ** 2 / (var ** 2).sum()

def stratum_test(idx, target_vals):
    """Mantel rho + p for one target within one stratum (graph indices idx)."""
    ed = pdist(vectors[idx], metric='cityblock')
    td = pdist(target_vals[idx].reshape(-1, 1), metric='cityblock')
    if td.max() == 0:                       # target constant within stratum
        return None, None
    rho, _ = spearmanr(ed, td)
    p = mantel_p(ed, target_vals[idx], rho, n_perm=N_PERM_STRATUM)
    return rho, p

def fisher_combine(pvals):
    pvals = [p for p in pvals if p is not None]
    stat = -2 * np.sum(np.log(pvals))
    return chi2.sf(stat, df=2 * len(pvals)), len(pvals)

stratum_results = {}
for label, members in structure_key_dict.items():
    if not label.startswith('tri_') or len(members) < MIN_STRATUM:
        continue
    idx = np.array(sorted(members))
    rho4, p4 = stratum_test(idx, num_4_cycles)
    rho5, p5 = stratum_test(idx, num_5_cycles)
    er = effective_rank(vectors[idx])
    stratum_results[label] = {'n': len(idx), 'rho_C4': rho4, 'p_C4': p4,
                              'rho_C5': rho5, 'p_C5': p5, 'eff_rank': er}
    print(f'{label:>7} (n={len(idx):3d}):  '
          f'C4 rho={rho4:+.3f} p={p4:.4f}   '
          f'C5 rho={rho5:+.3f} p={p5:.4f}   '
          f'eff_rank={er:.2f}')

p_global_C4, k4 = fisher_combine([v['p_C4'] for v in stratum_results.values()])
p_global_C5, k5 = fisher_combine([v['p_C5'] for v in stratum_results.values()])
print(f'\nFisher combined across {k4} strata:  '
      f'C4 p = {p_global_C4:.2e},  C5 p = {p_global_C5:.2e}')

  tri_0 (n=110):  C4 rho=+0.968 p=0.0001   C5 rho=+0.320 p=0.0001   eff_rank=2.90
  tri_1 (n=122):  C4 rho=+0.954 p=0.0001   C5 rho=+0.179 p=0.0001   eff_rank=2.88
  tri_2 (n=139):  C4 rho=+0.926 p=0.0001   C5 rho=+0.200 p=0.0001   eff_rank=3.06
  tri_3 (n= 74):  C4 rho=+0.906 p=0.0001   C5 rho=+0.175 p=0.0008   eff_rank=2.87
  tri_4 (n= 45):  C4 rho=+0.898 p=0.0001   C5 rho=+0.203 p=0.0019   eff_rank=2.79

Fisher combined across 5 strata:  C4 p = 2.05e-15,  C5 p = 1.98e-13


In [8]:
# The organization question, not just correlation: inside the largest strata,
# which axis is PC1 now? If C4 aligns with the leading within-stratum PC,
# the hierarchy claim is direct.

big_strata = sorted(stratum_results, key=lambda l: -stratum_results[l]['n'])[:4]

for label in big_strata:
    idx = np.array(sorted(structure_key_dict[label]))
    Xc = vectors[idx] - vectors[idx].mean(axis=0)
    U, S, Vt = svd(Xc, full_matrices=False)
    var_ratio = S**2 / (S**2).sum()
    scores = Xc @ Vt.T
    print(f'\n{label} (n={len(idx)}):')
    for pc in range(3):
        a4 = spearmanr(scores[:, pc], num_4_cycles[idx]).statistic
        a5 = spearmanr(scores[:, pc], num_5_cycles[idx]).statistic
        print(f'  PC{pc+1} (var {var_ratio[pc]:.3f}):  C4 {a4:+.3f}   C5 {a5:+.3f}')


tri_2 (n=139):
  PC1 (var 0.533):  C4 -0.978   C5 +0.170
  PC2 (var 0.161):  C4 +0.028   C5 -0.007
  PC3 (var 0.098):  C4 -0.040   C5 +0.169

tri_1 (n=122):
  PC1 (var 0.552):  C4 -0.981   C5 +0.353
  PC2 (var 0.156):  C4 -0.102   C5 +0.221
  PC3 (var 0.105):  C4 -0.026   C5 +0.086

tri_0 (n=110):
  PC1 (var 0.554):  C4 -0.988   C5 +0.476
  PC2 (var 0.141):  C4 +0.144   C5 -0.093
  PC3 (var 0.107):  C4 +0.006   C5 -0.121

tri_3 (n=74):
  PC1 (var 0.561):  C4 -0.981   C5 -0.096
  PC2 (var 0.138):  C4 -0.049   C5 +0.069
  PC3 (var 0.096):  C4 -0.012   C5 -0.021


In [9]:
# Second layer of the onion: joint strata from the count arrays directly.
# Strata shrink fast — report everything above threshold, expect few.

MIN_STRATUM_L2 = 12

joint = {}
for k in keys:
    joint.setdefault((num_triangles[k], num_4_cycles[k]), []).append(k)

l2_pvals = []
print(f'{"(tri,C4)":>10}   n     C5 rho     p')
for (t, q), members in sorted(joint.items()):
    if len(members) < MIN_STRATUM_L2:
        continue
    idx = np.array(members)
    rho5, p5 = stratum_test(idx, num_5_cycles)
    if rho5 is None:
        continue
    l2_pvals.append(p5)
    print(f'  ({t:2d},{q:2d})   {len(idx):3d}   {rho5:+.3f}   {p5:.4f}')

if l2_pvals:
    p_global, k = fisher_combine(l2_pvals)
    print(f'\nFisher combined across {k} joint strata:  C5 p = {p_global:.2e}')
else:
    print('\nNo joint strata above threshold — increase n or lower MIN_STRATUM_L2.')

  (tri,C4)   n     C5 rho     p
  ( 0, 1)    12   +0.926   0.0001
  ( 0, 2)    24   +0.871   0.0001
  ( 0, 3)    20   +0.747   0.0001
  ( 0, 4)    18   +0.521   0.0001
  ( 1, 1)    21   +0.870   0.0001
  ( 1, 2)    33   +0.893   0.0001
  ( 1, 3)    25   +0.470   0.0001
  ( 1, 4)    19   +0.316   0.0030
  ( 2, 1)    26   +0.836   0.0001
  ( 2, 2)    32   +0.637   0.0001
  ( 2, 3)    34   +0.417   0.0001
  ( 2, 4)    16   +0.434   0.0005
  ( 3, 1)    14   +0.581   0.0011
  ( 3, 2)    20   +0.363   0.0001
  ( 3, 3)    14   +0.737   0.0001
  ( 4, 3)    12   +0.518   0.0021

Fisher combined across 16 joint strata:  C5 p = 3.31e-40
